# CoDeC Publication Tables and Figures

This notebook creates publication-ready tables and figures for the three CoDeC
models used in the analysis: Qwen, Kimi, and Pythia. It reads compact
question-level and context-draw results from Supabase and does not download the
large token-level table.


## Setup

In [ ]:
%pip install -q supabase

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import Markdown, display
from supabase import create_client


## Configuration

In [ ]:
RUNS = {
    "Qwen": {
        "run_id": "20260513_201835_qwen3_vl_30b_a3b_thinking_7c2864e3",
        "label": "Qwen3-VL 30B-A3B",
        "color": "#4C78A8",
    },
    "Kimi": {
        "run_id": "20260728_121212_kimi_k2p6_388cd1ea",
        "label": "Kimi K2P6",
        "color": "#E39C37",
    },
    "Pythia": {
        "run_id": "20260728_181812_pythia_1p4b_deduped_d9a27f9b",
        "label": "Pythia 1.4B",
        "color": "#59A14F",
    },
}

OUTPUT_DIR = Path("outputs/figures")

supabase_url = os.environ.get("SUPABASE_URL")
supabase_key = os.environ.get("SUPABASE_KEY")
if not supabase_url or not supabase_key:
    raise ValueError("Missing Supabase environment variables.")

db = create_client(supabase_url, supabase_key)


## Load Results

Each run is loaded independently from the saved question-level results.


In [ ]:
model_frames = []

for model, settings in RUNS.items():
    results = pd.DataFrame(
        db.table("codec_question_results")
        .select(
            "question_id,season,mean_delta,delta_std,"
            "is_contaminated,n_seed_evals"
        )
        .eq("run_id", settings["run_id"])
        .order("question_id")
        .limit(1000)
        .execute()
        .data
        or []
    )

    if results.empty:
        raise ValueError(f"No question-level results were returned for {model}.")
    if results["question_id"].duplicated().any():
        raise ValueError(f"{model} contains duplicate question results.")
    if not results["n_seed_evals"].eq(5).all():
        raise ValueError(f"{model} contains incomplete context draws.")
    if not results["is_contaminated"].eq(results["mean_delta"] < 0).all():
        raise ValueError(f"{model} contains inconsistent contamination labels.")

    results["model"] = model
    model_frames.append(results)

comparison_results = pd.concat(model_frames, ignore_index=True)
print("Loaded question-level results for Qwen, Kimi, and Pythia.")


## Table 1: Cross-Model Comparison


In [ ]:
summary_rows = []

for model, settings in RUNS.items():
    model_results = comparison_results.loc[
        comparison_results["model"].eq(model)
    ]
    summary_rows.append(
        {
            "Model": settings["label"],
            "Contaminated questions": int(
                model_results["is_contaminated"].sum()
            ),
            "CoDeC score (%)": 100
            * model_results["is_contaminated"].mean(),
            "Mean delta": model_results["mean_delta"].mean(),
            "Std.": model_results["mean_delta"].std(ddof=0),
        }
    )

appendix_table = pd.DataFrame(summary_rows)

display(
    appendix_table.style
    .hide(axis="index")
    .format(
        {
            "Contaminated questions": "{:.0f}",
            "CoDeC score (%)": "{:.1f}",
            "Mean delta": "{:.3f}",
            "Std.": "{:.3f}",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("font-family", "Times New Roman, serif"),
                    ("font-size", "14px"),
                    ("line-height", "1.35"),
                ],
            },
            {
                "selector": "thead th",
                "props": [
                    ("border-top", "1.5px solid #222222"),
                    ("border-bottom", "0.8px solid #222222"),
                    ("padding", "7px 14px"),
                    ("text-align", "right"),
                    ("font-weight", "bold"),
                ],
            },
            {
                "selector": "thead th:first-child",
                "props": [("text-align", "left")],
            },
            {
                "selector": "tbody td",
                "props": [
                    ("border", "none"),
                    ("padding", "7px 14px"),
                    ("text-align", "right"),
                ],
            },
            {
                "selector": "tbody td:first-child",
                "props": [("text-align", "left")],
            },
            {
                "selector": "tbody tr:last-child td",
                "props": [("border-bottom", "1.5px solid #222222")],
            },
        ]
    )
)

display(
    Markdown(
        '<p style="font-size:1.05em; line-height:1.55; '
        'margin-top:14px; max-width:1050px;">'
        "<strong>Table 1 | CoDeC results by model, pooled across all "
        "seasons.</strong> For each question, delta is the change in the mean "
        "log probability of the target tokens after a same-season context "
        "question is added (with context minus without context), averaged "
        "over five context draws. A question is classified as contaminated "
        "when its delta is negative. Contaminated questions reports the "
        "number of such questions, CoDeC score reports their percentage, "
        "Mean delta reports the average across questions, and Std. reports "
        "the standard deviation.</p>"
    )

)


## Figure 1: Qwen Question-Level Deltas by Season


In [ ]:
plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 12,
        "axes.titlesize": 12,
        "axes.labelsize": 12,
        "axes.spines.top": True,
        "axes.spines.right": True,
        "axes.edgecolor": "#777777",
        "axes.linewidth": 0.8,
        "figure.dpi": 120,
        "savefig.dpi": 300,
    }
)

THRESHOLD_COLOR = "#A33A3A"
GRID_COLOR = "#E5E5E5"


def plot_season_distributions(axis, data, color, title=None):
    """Plot question-level delta distributions for one model."""
    seasons = sorted(data["season"].unique())
    distributions = [
        data.loc[data["season"].eq(season), "mean_delta"].to_numpy()
        for season in seasons
    ]
    positions = np.arange(1, len(seasons) + 1)

    axis.boxplot(
        distributions,
        positions=positions,
        widths=0.52,
        patch_artist=True,
        showfliers=False,
        boxprops={
            "facecolor": color,
            "edgecolor": color,
            "alpha": 0.20,
            "linewidth": 1.0,
        },
        medianprops={"color": color, "linewidth": 1.6},
        whiskerprops={"color": "#666666", "linewidth": 0.8},
        capprops={"color": "#666666", "linewidth": 0.8},
    )

    jitter = np.random.default_rng(42)
    for position, values in zip(positions, distributions):
        axis.scatter(
            jitter.normal(position, 0.05, len(values)),
            values,
            s=11,
            color=color,
            alpha=0.42,
            linewidth=0,
        )

    axis.axhline(0, color=THRESHOLD_COLOR, linewidth=1.0)
    axis.set_xticks(positions, [str(season) for season in seasons])
    axis.set_xlabel("Season")
    if title:
        axis.set_title(title)
    axis.grid(axis="y", color=GRID_COLOR, linewidth=0.7)


qwen_results = comparison_results.loc[
    comparison_results["model"].eq("Qwen")
].copy()

fig_qwen, ax_qwen = plt.subplots(figsize=(8.2, 4.7))
plot_season_distributions(
    ax_qwen,
    qwen_results,
    RUNS["Qwen"]["color"],
)
ax_qwen.set_ylabel("Question-level delta")
fig_qwen.tight_layout()
plt.show()

qwen_minimum = qwen_results["mean_delta"].min()
qwen_maximum = qwen_results["mean_delta"].max()
display(
    Markdown(
        '<p style="font-size:1.05em; line-height:1.55; '
        'margin-top:14px; max-width:1050px;">'
        "<strong>Figure 1 | Distribution of question-level CoDeC deltas by "
        f"season for {RUNS['Qwen']['label']}.</strong> Each point represents "
        "one question and shows the mean delta across five same-season "
        "context draws. Positive values indicate that context increased the "
        "model's confidence in the target question; negative values indicate "
        "a contamination signal. Boxes show the median and interquartile "
        "range, and whiskers extend to 1.5 times the interquartile range. "
        f"Deltas ranged from {qwen_minimum:.3f} to {qwen_maximum:.3f}. "
        "All deltas were positive, and no question was classified as "
        "contaminated.</p>"
    )
)


## Figure 2: Reference-Based Comparison


In [ ]:
models = list(RUNS)
global_min = comparison_results["mean_delta"].min()
global_max = comparison_results["mean_delta"].max()
y_padding = 0.06 * (global_max - min(0, global_min))
y_limits = (
    min(-0.01, global_min - y_padding),
    global_max + y_padding,
)

fig_reference, axes = plt.subplots(
    1,
    len(models),
    figsize=(13.2, 4.5),
    sharex=True,
    sharey=True,
)

for axis, model in zip(axes, models):
    model_data = comparison_results.loc[
        comparison_results["model"].eq(model)
    ]
    plot_season_distributions(
        axis,
        model_data,
        RUNS[model]["color"],
        RUNS[model]["label"],
    )
    axis.set_ylim(y_limits)

axes[0].set_ylabel("Average change in target logprob (delta)")
fig_reference.tight_layout()
plt.show()

display(
    Markdown(
        '<p style="font-size:1.05em; line-height:1.55; '
        'margin-top:14px; max-width:1200px;">'
        "<strong>Figure 2 | Distribution of question-level CoDeC deltas by "
        "season and model.</strong> Each point represents one question and "
        "shows the mean delta across five same-season context draws. All "
        "deltas were positive for Qwen3-VL 30B-A3B, Kimi K2P6, and Pythia "
        "1.4B, indicating that context increased model confidence and that "
        "no question was classified as contaminated.</p>"
    )
)


## Figure 3: Season-Level Patterns Across Models


In [ ]:
season_means = (
    comparison_results.groupby(["season", "model"], as_index=False)
    .agg(mean_delta=("mean_delta", "mean"))
)

fig_profiles, ax_profiles = plt.subplots(figsize=(7.4, 4.5))
for model, settings in RUNS.items():
    values = season_means.loc[season_means["model"].eq(model)]
    ax_profiles.plot(
        values["season"],
        values["mean_delta"],
        marker="o",
        linewidth=1.8,
        markersize=5.2,
        color=settings["color"],
        label=settings["label"],
    )

ax_profiles.axhline(0, color=THRESHOLD_COLOR, linewidth=1.0)
ax_profiles.set_xticks(sorted(season_means["season"].unique()))
ax_profiles.set_xlabel("Season")
ax_profiles.set_ylabel("Season-level delta")
ax_profiles.legend(
    frameon=False,
    ncol=3,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.01),
    fontsize=10,
)
ax_profiles.grid(axis="y", color=GRID_COLOR, linewidth=0.7)
fig_profiles.tight_layout()
plt.show()

display(
    Markdown(
        '<p style="font-size:1.05em; line-height:1.55; '
        'margin-top:14px; max-width:1050px;">'
        "<strong>Figure 3 | Mean CoDeC delta by season and model.</strong> "
        "Each point shows the average question-level delta for one season. "
        "The similar profiles indicate that seasonal differences were "
        "broadly consistent across the three models.</p>"
    )
)


## Save Tables and Figures


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

appendix_table.to_csv(
    OUTPUT_DIR / "codec_model_summary.csv",
    index=False,
)
appendix_table.to_latex(
    OUTPUT_DIR / "codec_model_summary.tex",
    index=False,
    column_format="lrrrr",
    formatters={
        "Contaminated questions": "{:.0f}".format,
        "CoDeC score (%)": "{:.1f}".format,
        "Mean delta": "{:.3f}".format,
        "Std.": "{:.3f}".format,
    },
    caption="CoDeC results across models, with all seasons pooled.",
    label="tab:codec-model-results",
)

figures = {
    "qwen_codec_season_distributions": fig_qwen,
    "codec_reference_based_comparison": fig_reference,
    "codec_season_level_profiles": fig_profiles,
}

for name, figure in figures.items():
    for extension in ("png", "pdf", "svg"):
        figure.savefig(
            OUTPUT_DIR / f"{name}.{extension}",
            dpi=300,
            bbox_inches="tight",
        )

print(f"Saved publication outputs to {OUTPUT_DIR.resolve()}")
